In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_absolute_error
from sklearn.impute import SimpleImputer

df = pd.read_csv('/content/bengaluru_house_prices (1).csv')

def convert_sqft(x):
    try:
        if '-' in str(x):
            vals = x.split('-')
            return (float(vals[0]) + float(vals[1])) / 2
        return float(x)
    except:
        return np.nan

df['total_sqft'] = df['total_sqft'].apply(convert_sqft)
df = df.drop_duplicates()

def extract_bhk(size):
    try:
        if 'BHK' in size:
            return int(size.split(' ')[0])
        elif 'Bedroom' in size:
            return int(size.split(' ')[0])
        else:
            return np.nan
    except:
        return np.nan

df['bhk'] = df['size'].apply(extract_bhk)

df = df[['price', 'total_sqft', 'bath', 'bhk', 'location']].copy()

df = pd.get_dummies(df, columns=['location'], drop_first=True)

imputer = SimpleImputer(strategy='median')
df[df.columns] = imputer.fit_transform(df)

df['price_per_sqft'] = df['price'] / df['total_sqft']
df = df[(df['price_per_sqft'] > df['price_per_sqft'].quantile(0.05)) &
        (df['price_per_sqft'] < df['price_per_sqft'].quantile(0.95))]

X = df.drop(['price', 'price_per_sqft'], axis=1)
y = df['price']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

model = LinearRegression()
model.fit(X_train, y_train)
y_pred = model.predict(X_test)

r2 = r2_score(y_test, y_pred)
mae = mean_absolute_error(y_test, y_pred)
print(f"linearRegression R2:{r2}MAE:{mae}")

print(f"R² = {r2:.4f}  ({r2*100:.1f}%)")
print(f"MAE = {mae:.2f}")

/tmp/ipykernel_1230/1026019516.py:42: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df['price_per_sqft'] = df['price'] / df['total_sqft']


linearRegression R2:0.7601286890208351MAE:25.557308555151327
R² = 0.7601  (76.0%)
MAE = 25.56
